In [ ]:
!pip install -q -U transformers accelerate
!pip install -q peft trl bitsandbytes datasets scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 55.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

filepath = "/content/drive/MyDrive/Colab Notebooks/output.jsonl"

data = []
with open(filepath, "r") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(data)
print(f"Total loaded: {len(df)}")
print(df['label'].value_counts())

Total loaded: 74184
label
benign      39284
phishing    34900
Name: count, dtype: int64


In [ ]:
SYSTEM_PROMPT = """You are a phishing detection model. You analyze structured data extracted from web pages and determine whether a URL is benign or phishing.

You will receive a JSON object containing:
- url: parsed URL features including scheme, hostname, registered domain, detected brands, and brand-domain mismatches
- redirects: redirect chain analysis including cross-domain redirects and domain changes
- html: page content including title, visible text, forms with input fields, anchors, and iframes
- http: HTTP response metadata including status code, content type, and server

Analyze all the provided data holistically. Consider URL structure, page content, form behavior, redirect patterns, and any brand impersonation indicators.

Respond in this exact format:
Analysis:
- [observation 1]
- [observation 2]
- ...
Label: [benign or phishing]"""


def build_user_prompt(row):
    """Build the user message from raw features (no signals — model must learn to derive them)."""
    features = {
        "url": row["url"],
        "redirects": row["redirects"],
        "html": row["html"],
        "http": row["http"],
    }
    return json.dumps(features, indent=2, default=str)


def build_assistant_response(row):
    """Build the target response using signal statements as the analysis."""
    signals = row["signals"]
    label = row["label"]

    # Build analysis from signal statements
    analysis_lines = []
    for sig in signals:
        analysis_lines.append(f"- {sig['statement']}")

    # If no signals, add a generic observation
    if not analysis_lines:
        analysis_lines.append("- No significant indicators were detected in the page data.")

    analysis = "\n".join(analysis_lines)
    return f"Analysis:\n{analysis}\nLabel: {label}"


def format_example(row):
    """Create the full chat-format training example."""
    return {
        "system": SYSTEM_PROMPT,
        "user": build_user_prompt(row),
        "assistant": build_assistant_response(row),
        "label": row["label"],
    }


# Test with one example
sample = df.iloc[0]
example = format_example(sample)
print("=== SYSTEM ===")
print(example["system"][:200], "...")
print("\n=== USER (first 500 chars) ===")
print(example["user"][:500], "...")
print("\n=== ASSISTANT ===")
print(example["assistant"])

=== SYSTEM ===
You are a phishing detection model. You analyze structured data extracted from web pages and determine whether a URL is benign or phishing.

You will receive a JSON object containing:
- url: parsed UR ...

=== USER (first 500 chars) ===
{
  "url": {
    "raw": "https://ncls1.com/",
    "scheme": "https",
    "hostname": "ncls1.com",
    "registered_domain": "ncls1.com",
    "path": "/",
    "query": "",
    "uses_http_not_https": false,
    "uses_ip_address": false,
    "contains_suspicious_terms": [],
    "detected_brands": [],
    "brand_domain_mismatches": []
  },
  "redirects": {
    "requested_url": "https://ncls1.com/",
    "final_url": "https://ncls1.com/",
    "redirect_count": 0,
    "chain": [],
    "initial_registere ...

=== ASSISTANT ===
Analysis:
- No urgent, security-incident, or account-verification language was detected in the visible text.
- No password, email, or phone input fields were detected in the page forms.
- No cross-domain redirect behavior wa

In [ ]:
# Use 4,000 samples: 2k benign + 2k phishing (balanced)
N_PER_CLASS = 2000

benign_df = df[df['label'] == 'benign'].sample(n=N_PER_CLASS, random_state=42)
phishing_df = df[df['label'] == 'phishing'].sample(n=N_PER_CLASS, random_state=42)
subset = pd.concat([benign_df, phishing_df]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Subset: {len(subset)}")
print(subset['label'].value_counts())

# Split: 90% train, 10% test
train_df, test_df = train_test_split(subset, test_size=0.1, stratify=subset['label'], random_state=42)
print(f"\nTrain: {len(train_df)} | Test: {len(test_df)}")
print(f"Train:\n{train_df['label'].value_counts()}")
print(f"Test:\n{test_df['label'].value_counts()}")

# Format all examples
train_examples = [format_example(row) for _, row in train_df.iterrows()]
test_examples = [format_example(row) for _, row in test_df.iterrows()]
print(f"\n✅ Train examples: {len(train_examples)} | Test examples: {len(test_examples)}")

Subset: 4000
label
benign      2000
phishing    2000
Name: count, dtype: int64

Train: 3600 | Test: 400
Train:
label
phishing    1800
benign      1800
Name: count, dtype: int64
Test:
label
benign      200
phishing    200
Name: count, dtype: int64

✅ Train examples: 3600 | Test examples: 400


In [ ]:
import torch
import gc
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Store all results
all_results = {}


def cleanup():
    """Free GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 GPU memory cleared")


def load_model_and_tokenizer(model_name):
    """Load a model in 4-bit quantization."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    return model, tokenizer


def run_inference(model, tokenizer, examples, max_new_tokens=300, batch_desc="Evaluating"):
    """Run inference on test examples and extract predictions."""
    model.eval()
    predictions = []
    raw_outputs = []

    for i, ex in enumerate(examples):
        if i % 50 == 0:
            print(f"  {batch_desc}: {i}/{len(examples)}...")

        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
        ]

        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=3072).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
            )

        # Decode only the new tokens
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        raw_outputs.append(response)

        # Extract label from response
        response_lower = response.lower()
        if "label: phishing" in response_lower or "label:phishing" in response_lower:
            predictions.append("phishing")
        elif "label: benign" in response_lower or "label:benign" in response_lower:
            predictions.append("benign")
        elif "phishing" in response_lower.split("\n")[-1]:
            predictions.append("phishing")
        elif "benign" in response_lower.split("\n")[-1]:
            predictions.append("benign")
        else:
            predictions.append("unknown")

    return predictions, raw_outputs


def evaluate_predictions(true_labels, predictions, model_name, phase):
    """Calculate and display metrics."""
    # Filter out unknowns for metrics
    valid = [(t, p) for t, p in zip(true_labels, predictions) if p != "unknown"]
    unknown_count = len(true_labels) - len(valid)

    if valid:
        t_valid, p_valid = zip(*valid)
    else:
        print(f"⚠️ No valid predictions for {model_name} ({phase})")
        return None

    metrics = {
        "model": model_name,
        "phase": phase,
        "accuracy": accuracy_score(t_valid, p_valid),
        "precision": precision_score(t_valid, p_valid, pos_label="phishing", zero_division=0),
        "recall": recall_score(t_valid, p_valid, pos_label="phishing", zero_division=0),
        "f1": f1_score(t_valid, p_valid, pos_label="phishing", zero_division=0),
        "unknown": unknown_count,
        "total": len(true_labels),
    }

    print(f"\n{'='*60}")
    print(f"📊 {model_name} — {phase}")
    print(f"{'='*60}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1']:.4f}")
    print(f"  Unknown:   {unknown_count}/{len(true_labels)}")
    print(f"\n{classification_report(t_valid, p_valid, digits=4)}")
    return metrics

In [ ]:
def fine_tune_model(model_name, short_name, train_examples, tokenizer_ref=None):
    """Fine-tune a model with LoRA and return the model + tokenizer."""
    print(f"\n{'#'*60}")
    print(f"# FINE-TUNING: {short_name}")
    print(f"{'#'*60}")

    model, tokenizer = load_model_and_tokenizer(model_name)

    # Format training data with chat template
    def format_for_training(ex):
        messages = [
            {"role": "system", "content": ex["system"]},
            {"role": "user", "content": ex["user"]},
            {"role": "assistant", "content": ex["assistant"]},
        ]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

    train_formatted = [format_for_training(ex) for ex in train_examples]
    train_dataset = Dataset.from_list(train_formatted)

    # Prepare for LoRA
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

    trainable, total = model.get_nb_trainable_parameters()
    print(f"  Total params:     {total:>12,}")
    print(f"  Trainable params: {trainable:>12,} ({100 * trainable / total:.2f}%)")

    # Training config
    training_args = SFTConfig(
        output_dir=f"/content/{short_name}-finetuned",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=50,
        save_strategy="no",
        bf16=True,
        dataset_text_field="text",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
    )

    print(f"\n🚀 Training {short_name}...")
    start = time.time()
    trainer.train()
    train_time = time.time() - start
    print(f"✅ {short_name} training complete in {train_time/60:.1f} minutes")

    return model, tokenizer, train_time

In [ ]:
from huggingface_hub import login
login(token="HUGGINGFACE_TOKEN_REMOVED")

In [ ]:
BASELINE_N = 300
baseline_test = test_examples[:BASELINE_N]
true_labels_baseline = [ex["label"] for ex in baseline_test]

short_name = "Llama-3.2-3B"
model_name = "meta-llama/Llama-3.2-3B-Instruct"

print(f"\n{'#'*60}")
print(f"# BASELINE: {short_name}")
print(f"{'#'*60}")

model, tokenizer = load_model_and_tokenizer(model_name)

start = time.time()
preds, raws = run_inference(model, tokenizer, baseline_test, batch_desc=f"Baseline {short_name}")
elapsed = time.time() - start

metrics = evaluate_predictions(true_labels_baseline, preds, short_name, "Baseline")
if metrics:
    metrics["time_seconds"] = elapsed
    all_results[f"{short_name}_baseline"] = metrics

print(f"\n--- Sample outputs ({short_name} baseline) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_baseline[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:300]}...")

del model, tokenizer
cleanup()


############################################################
# BASELINE: Llama-3.2-3B
############################################################


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

  Baseline Llama-3.2-3B: 0/300...


KeyboardInterrupt: 

In [ ]:
short_name = "Llama-3.1-8B"
model_name = "meta-llama/Llama-3.1-8B-Instruct"

print(f"\n{'#'*60}")
print(f"# BASELINE: {short_name}")
print(f"{'#'*60}")

model, tokenizer = load_model_and_tokenizer(model_name)

start = time.time()
preds, raws = run_inference(model, tokenizer, baseline_test, batch_desc=f"Baseline {short_name}")
elapsed = time.time() - start

metrics = evaluate_predictions(true_labels_baseline, preds, short_name, "Baseline")
if metrics:
    metrics["time_seconds"] = elapsed
    all_results[f"{short_name}_baseline"] = metrics

print(f"\n--- Sample outputs ({short_name} baseline) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_baseline[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:300]}...")

del model, tokenizer
cleanup()

In [13]:
short_name = "Llama-3.2-3B"
model_name = "meta-llama/Llama-3.2-3B-Instruct"

model, tokenizer, train_time = fine_tune_model(model_name, short_name, train_examples)

true_labels_test = [ex["label"] for ex in test_examples]
start = time.time()
preds, raws = run_inference(model, tokenizer, test_examples, batch_desc=f"Fine-tuned {short_name}")
eval_time = time.time() - start

metrics = evaluate_predictions(true_labels_test, preds, short_name, "Fine-tuned")
if metrics:
    metrics["train_time_min"] = train_time / 60
    metrics["eval_time_seconds"] = eval_time
    all_results[f"{short_name}_finetuned"] = metrics

print(f"\n--- Sample outputs ({short_name} fine-tuned) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_test[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:500]}...")

del model, tokenizer
cleanup()


############################################################
# FINE-TUNING: Llama-3.2-3B
############################################################


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Total params:     3,237,063,680
  Trainable params:   24,313,856 (0.75%)


Adding EOS to train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

...

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.



🚀 Training Llama-3.2-3B...


Step,Training Loss
50,1.498571
100,0.956378
150,0.956979
200,3.559438
250,5.506403
300,6.164667
350,6.231169
400,6.243183
450,6.259307
500,6.035211


✅ Llama-3.2-3B training complete in 61.7 minutes
  Fine-tuned Llama-3.2-3B: 0/400...
  Fine-tuned Llama-3.2-3B: 50/400...


KeyboardInterrupt: 

In [ ]:
short_name = "Llama-3.1-8B"
model_name = "meta-llama/Llama-3.1-8B-Instruct"

model, tokenizer, train_time = fine_tune_model(model_name, short_name, train_examples)

true_labels_test = [ex["label"] for ex in test_examples]
start = time.time()
preds, raws = run_inference(model, tokenizer, test_examples, batch_desc=f"Fine-tuned {short_name}")
eval_time = time.time() - start

metrics = evaluate_predictions(true_labels_test, preds, short_name, "Fine-tuned")
if metrics:
    metrics["train_time_min"] = train_time / 60
    metrics["eval_time_seconds"] = eval_time
    all_results[f"{short_name}_finetuned"] = metrics

print(f"\n--- Sample outputs ({short_name} fine-tuned) ---")
for i in range(3):
    print(f"\nTrue: {true_labels_test[i]} | Predicted: {preds[i]}")
    print(f"Output: {raws[i][:500]}...")

del model, tokenizer
cleanup()


############################################################
# FINE-TUNING: Llama-3.1-8B
############################################################


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Tokenizing train dataset:   0%|          | 0/3600 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.



🚀 Training Llama-3.1-8B...


Step,Training Loss
50,1.234505
100,0.996358
150,3.872006
200,6.401616
250,6.336524
300,6.094586
350,6.154103
400,5.994630
450,5.801324
500,5.695160


✅ Llama-3.1-8B training complete in 109.2 minutes
  Fine-tuned Llama-3.1-8B: 0/400...
  Fine-tuned Llama-3.1-8B: 50/400...
  Fine-tuned Llama-3.1-8B: 100/400...


In [ ]:
results_df = pd.DataFrame(all_results).T
print("\n" + "="*80)
print("📊 FULL RESULTS COMPARISON")
print("="*80)
print(results_df[["model", "phase", "accuracy", "precision", "recall", "f1", "unknown"]].to_string())